In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# LOAD RANDOM RAW ROWS FROM ALL YEARS
# Memory optimized
# Does not save files
# Shows random rows for every year found
# Prints progress so it does not look frozen
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 100_000
ROWS_PER_YEAR = 3

# If True, show every raw column from the file.
# If False, show only useful columns listed below.
SHOW_ALL_COLUMNS = False

# Set to 42 if you want same random rows every time.
# Leave as None if you want different random rows each run.
RANDOM_SEED = None

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 150)


# ------------------------------------------------------------
# Check file exists
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file:\n{path}")
    print("FOUND:", path)


# ------------------------------------------------------------
# Read column names only
# ------------------------------------------------------------
def get_columns(path):
    return list(pd.read_csv(path, nrows=0).columns)


# ------------------------------------------------------------
# Find column by name safely, ignoring uppercase/lowercase
# ------------------------------------------------------------
def find_col(all_cols, wanted):
    lookup = {str(c).strip().lower(): c for c in all_cols}
    return lookup.get(str(wanted).strip().lower(), None)


# ------------------------------------------------------------
# Pick columns that actually exist
# ------------------------------------------------------------
def pick_existing_columns(all_cols, wanted_cols):
    found = []
    seen = set()

    for wanted in wanted_cols:
        real_col = find_col(all_cols, wanted)
        if real_col is not None and real_col not in seen:
            found.append(real_col)
            seen.add(real_col)

    return found


# ------------------------------------------------------------
# Check if row has useful values
# Keeps rows where at least one chosen value column is not blank
# and not all numeric zero
# ------------------------------------------------------------
def has_any_useful_value(df, cols):
    cols = [c for c in cols if c in df.columns]

    if not cols:
        return pd.Series(True, index=df.index)

    mask = pd.Series(False, index=df.index)

    for col in cols:
        s = df[col]

        text = s.astype(str).str.strip()
        text_good = (
            s.notna()
            & (text != "")
            & (text.str.lower() != "nan")
            & (text.str.lower() != "none")
            & (text.str.lower() != "null")
        )

        num = pd.to_numeric(s, errors="coerce")
        numeric_good = num.notna() & (num != 0)

        # Keep if text exists OR numeric value is nonzero
        mask = mask | text_good | numeric_good

    return mask


# ------------------------------------------------------------
# Main random loader
# This scans file by chunks and keeps only small random samples
# ------------------------------------------------------------
def random_raw_rows_all_years(
    path,
    dataset_name,
    rows_per_year=3,
    chunksize=100_000,
    year_col="year",
    show_cols=None,
    value_check_cols=None,
    show_all_columns=False,
    random_seed=None
):
    print("\n" + "=" * 120)
    print(dataset_name)
    print("=" * 120)

    check_file(path)

    all_cols = get_columns(path)
    actual_year_col = find_col(all_cols, year_col)

    if actual_year_col is None:
        print("\nCOLUMNS FOUND:")
        display(pd.DataFrame({
            "column_number": range(1, len(all_cols) + 1),
            "column_name": all_cols
        }))
        raise KeyError(f"Year column '{year_col}' was not found in {dataset_name}")

    print("Total columns:", len(all_cols))
    print("Year column used:", actual_year_col)
    print("Rows per year requested:", rows_per_year)
    print("Chunk size:", chunksize)

    if show_cols is None:
        show_cols = []

    if value_check_cols is None:
        value_check_cols = []

    real_show_cols = pick_existing_columns(all_cols, show_cols)
    real_value_cols = pick_existing_columns(all_cols, value_check_cols)

    # Make sure year is always included
    if actual_year_col not in real_show_cols:
        real_show_cols = [actual_year_col] + real_show_cols

    if show_all_columns:
        read_cols = all_cols
        final_show_cols = all_cols
    else:
        read_cols = []
        for c in [actual_year_col] + real_show_cols + real_value_cols:
            if c in all_cols and c not in read_cols:
                read_cols.append(c)

        final_show_cols = [c for c in real_show_cols if c in read_cols]

    print("Columns loaded:", len(read_cols))

    if not show_all_columns:
        print("\nColumns that will display:")
        display(pd.DataFrame({
            "column_number": range(1, len(final_show_cols) + 1),
            "column_name": final_show_cols
        }))

    rng = np.random.default_rng(random_seed)

    samples_by_year = {}
    total_rows_scanned = 0
    total_rows_kept_after_filter = 0
    chunks_read = 0

    reader = pd.read_csv(
        path,
        usecols=read_cols,
        chunksize=chunksize,
        low_memory=False,
        on_bad_lines="skip"
    )

    for chunk in reader:
        chunks_read += 1
        total_rows_scanned += len(chunk)

        chunk[actual_year_col] = pd.to_numeric(chunk[actual_year_col], errors="coerce")
        chunk = chunk.dropna(subset=[actual_year_col])

        if chunk.empty:
            print(f"Chunk {chunks_read}: scanned {total_rows_scanned:,} rows | no valid year")
            continue

        chunk[actual_year_col] = chunk[actual_year_col].astype(int)

        # Keep rows with useful values
        if real_value_cols:
            useful_mask = has_any_useful_value(chunk, real_value_cols)
            chunk = chunk.loc[useful_mask].copy()

        total_rows_kept_after_filter += len(chunk)

        if chunk.empty:
            print(f"Chunk {chunks_read}: scanned {total_rows_scanned:,} rows | no useful rows after filter")
            continue

        # Random key lets us keep the best random rows per year
        chunk["_random_key"] = rng.random(len(chunk))

        # Keep only rows_per_year best random rows per year from this chunk
        for year, group in chunk.groupby(actual_year_col):
            group_small = group.nsmallest(rows_per_year, "_random_key").copy()

            if year not in samples_by_year:
                samples_by_year[year] = group_small
            else:
                combined = pd.concat(
                    [samples_by_year[year], group_small],
                    ignore_index=True
                )
                samples_by_year[year] = combined.nsmallest(rows_per_year, "_random_key").copy()

        print(
            f"Chunk {chunks_read}: scanned {total_rows_scanned:,} rows | "
            f"kept useful {total_rows_kept_after_filter:,} rows | "
            f"years found so far {len(samples_by_year)}"
        )

    print("\nDONE READING FILE")
    print("Chunks read:", chunks_read)
    print("Rows scanned:", f"{total_rows_scanned:,}")
    print("Useful rows kept/checkable:", f"{total_rows_kept_after_filter:,}")
    print("Years found:", len(samples_by_year))

    if not samples_by_year:
        print("No sample rows found.")
        return pd.DataFrame()

    result = pd.concat(
        [samples_by_year[y] for y in sorted(samples_by_year.keys())],
        ignore_index=True
    )

    result = result.drop(columns=["_random_key"], errors="ignore")

    # Sort by year so easy to check
    result = result.sort_values(actual_year_col).reset_index(drop=True)

    # Display only requested columns unless showing all columns
    if not show_all_columns:
        display_cols = [c for c in final_show_cols if c in result.columns]
        result_display = result[display_cols].copy()
    else:
        result_display = result.copy()

    print("\nRANDOM RAW ROWS FROM ALL YEARS:")
    display(result_display)

    print("\nYEAR COUNT IN SAMPLE:")
    year_count_table = (
        result_display[actual_year_col]
        .value_counts()
        .sort_index()
        .reset_index()
    )
    year_count_table.columns = ["year", "sample_rows"]
    display(year_count_table)

    print("\nYEAR RANGE FOUND:")
    print("First year:", int(result_display[actual_year_col].min()))
    print("Last year:", int(result_display[actual_year_col].max()))

    return result_display


# ============================================================
# DEGREE DATA SETTINGS
# ============================================================

degree_show_cols = [
    "year",
    "unitid",
    "UNITID",
    "cipcode",
    "CIPCODE",
    "cip_code",
    "awlevel",
    "AWLEVEL",
    "award_level_code",
    "award_level_name",
    "degree_group",
    "degree_count",
    "field_group",
    "field_subgroup",
    "major_name",
    "CTOTALT",
    "CTOTALM",
    "CTOTALW",
    "crace01",
    "crace02",
    "crace03",
    "crace04",
    "crace05",
    "crace06",
    "crace07",
    "crace08",
    "crace09",
    "crace10",
    "crace11",
    "crace12",
    "crace13",
    "crace14",
    "crace15",
    "crace16",
    "CRACE01",
    "CRACE02",
    "CRACE03",
    "CRACE04",
    "CRACE05",
    "CRACE06",
    "CRACE07",
    "CRACE08",
    "CRACE09",
    "CRACE10",
    "CRACE11",
    "CRACE12",
    "CRACE13",
    "CRACE14",
    "CRACE15",
    "CRACE16",
]

degree_value_check_cols = [
    "degree_count",
    "CTOTALT",
    "CTOTALM",
    "CTOTALW",
    "CNRALT",
    "CUNKNT",
    "CHISPT",
    "CAIANT",
    "CASIAT",
    "CBKAAT",
    "CNHPIT",
    "CWHITT",
    "C2MORT",
    "crace01",
    "crace02",
    "crace03",
    "crace04",
    "crace05",
    "crace06",
    "crace07",
    "crace08",
    "crace09",
    "crace10",
    "crace11",
    "crace12",
    "crace13",
    "crace14",
    "crace15",
    "crace16",
    "CRACE01",
    "CRACE02",
    "CRACE03",
    "CRACE04",
    "CRACE05",
    "CRACE06",
    "CRACE07",
    "CRACE08",
    "CRACE09",
    "CRACE10",
    "CRACE11",
    "CRACE12",
    "CRACE13",
    "CRACE14",
    "CRACE15",
    "CRACE16",
]


# ============================================================
# COMPANY DATA SETTINGS
# ============================================================

company_show_cols = [
    "source",
    "dataset",
    "source_table",
    "source_file",
    "year",
    "quarter",
    "month",
    "date",
    "geo_level",
    "geo_code",
    "geo_name",
    "state_code",
    "state_name",
    "county_code",
    "county_name",
    "msa_code",
    "msa_name",
    "metro_nonmetro",
    "naics_code",
    "industry_code",
    "industry_name",
    "firm_age",
    "establishment_age",
    "firm_size",
    "establishment_size",
    "metric_code",
    "metric_name",
    "metric_category",
    "value",
    "unit",
    "seasonal",
    "notes",
]

company_value_check_cols = [
    "value",
    "industry_name",
    "metric_name",
    "metric_category",
]


# ============================================================
# RUN DEGREE DATA
# ============================================================

degree_random_all_years = random_raw_rows_all_years(
    path=DEGREE_FILE,
    dataset_name="DEGREE DATA - RANDOM RAW ROWS FROM ALL YEARS",
    rows_per_year=ROWS_PER_YEAR,
    chunksize=CHUNKSIZE,
    year_col="year",
    show_cols=degree_show_cols,
    value_check_cols=degree_value_check_cols,
    show_all_columns=SHOW_ALL_COLUMNS,
    random_seed=RANDOM_SEED
)


# ============================================================
# RUN COMPANY DATA
# ============================================================

company_random_all_years = random_raw_rows_all_years(
    path=COMPANY_FILE,
    dataset_name="COMPANY DATA - RANDOM RAW ROWS FROM ALL YEARS",
    rows_per_year=ROWS_PER_YEAR,
    chunksize=CHUNKSIZE,
    year_col="year",
    show_cols=company_show_cols,
    value_check_cols=company_value_check_cols,
    show_all_columns=SHOW_ALL_COLUMNS,
    random_seed=RANDOM_SEED
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# RANDOM ROWS BY YEAR WITH REAL VALUES ONLY
# No save
# No edit original files
# Chunked for huge CSV files
# Removes NaN from display
# Keeps only rows where usable value > 0
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"
COMPANY_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"

CHUNKSIZE = 50_000
ROWS_PER_YEAR = 3
RANDOM_SEED = 42

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 300)
pd.set_option("display.max_colwidth", 120)


# ------------------------------------------------------------
# Basic helpers
# ------------------------------------------------------------
def check_file(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"FOUND: {path}")
    print(f"SIZE: {size_mb:,.2f} MB")


def get_columns(path):
    return list(pd.read_csv(path, nrows=0).columns)


def find_year_column(columns):
    possible = [
        "year", "Year", "YEAR",
        "survey_year", "academic_year",
        "report_year", "data_year"
    ]

    for col in possible:
        if col in columns:
            return col

    raise KeyError(
        "No year column found. Available columns are:\n"
        + "\n".join(columns)
    )


def to_number(df, col):
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce").fillna(0)
    return pd.Series(0, index=df.index, dtype="float64")


def first_nonblank(df, cols, default=""):
    out = pd.Series("", index=df.index, dtype="object")

    for col in cols:
        if col not in df.columns:
            continue

        s = df[col].astype("object").where(df[col].notna(), "")
        s = s.astype(str).str.strip()

        s = s.replace({
            "nan": "",
            "NaN": "",
            "None": "",
            "NONE": "",
            "<NA>": ""
        })

        mask = (out == "") & (s != "")
        out.loc[mask] = s.loc[mask]

    out = out.replace("", default)
    return out


def display_clean(df):
    clean = df.copy()
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.fillna("")
    display(clean)


# ------------------------------------------------------------
# Build compact degree rows
# ------------------------------------------------------------
def build_degree_compact(chunk):
    value = pd.Series(0.0, index=chunk.index)
    value_source = pd.Series("", index=chunk.index, dtype="object")

    # Best cleaned column if available
    degree_count = to_number(chunk, "degree_count")

    # Later IPEDS total column
    ctotalt = to_number(chunk, "CTOTALT") + to_number(chunk, "ctotalt")

    # Older IPEDS layout: male + female total often stored here
    old_lower_total = to_number(chunk, "crace15") + to_number(chunk, "crace16")
    old_upper_total = to_number(chunk, "CRACE15") + to_number(chunk, "CRACE16")

    fallback_values = [
        ("degree_count", degree_count),
        ("CTOTALT", ctotalt),
        ("crace15 + crace16", old_lower_total),
        ("CRACE15 + CRACE16", old_upper_total),
    ]

    for source_name, source_value in fallback_values:
        mask = (value <= 0) & (source_value > 0)
        value.loc[mask] = source_value.loc[mask]
        value_source.loc[mask] = source_name

    male_count = to_number(chunk, "CTOTALM") + to_number(chunk, "ctotalm")
    female_count = to_number(chunk, "CTOTALW") + to_number(chunk, "ctotalw")

    old_male = to_number(chunk, "crace15") + to_number(chunk, "CRACE15")
    old_female = to_number(chunk, "crace16") + to_number(chunk, "CRACE16")

    male_count = male_count.where(male_count > 0, old_male)
    female_count = female_count.where(female_count > 0, old_female)

    compact = pd.DataFrame({
        "year": chunk["__year_clean__"],
        "csv_row_number": chunk["__csv_row_number__"],

        "unitid": first_nonblank(chunk, ["unitid", "UNITID"], default="not listed"),
        "cip_code": first_nonblank(chunk, ["cip_code", "cipcode", "CIPCODE"], default="not listed"),

        "award_level_code": first_nonblank(chunk, ["award_level_code", "awlevel", "AWLEVEL"], default="not listed"),
        "award_level_name": first_nonblank(chunk, ["award_level_name"], default="not listed"),
        "degree_group": first_nonblank(chunk, ["degree_group"], default="not listed"),

        "field_group": first_nonblank(chunk, ["field_group"], default="not listed"),
        "field_subgroup": first_nonblank(chunk, ["field_subgroup"], default="not listed"),
        "major_name": first_nonblank(chunk, ["major_name"], default="not listed"),

        "usable_degree_value": value,
        "male_count": male_count,
        "female_count": female_count,
        "value_source": value_source.replace("", "not found")
    })

    compact["__sample_value__"] = compact["usable_degree_value"]

    return compact


# ------------------------------------------------------------
# Build compact company rows
# ------------------------------------------------------------
def build_company_compact(chunk):
    value = to_number(chunk, "value")

    compact = pd.DataFrame({
        "year": chunk["__year_clean__"],
        "csv_row_number": chunk["__csv_row_number__"],

        "source": first_nonblank(chunk, ["source"], default="not listed"),
        "dataset": first_nonblank(chunk, ["dataset"], default="not listed"),

        "geo_level": first_nonblank(chunk, ["geo_level"], default="not listed"),
        "state_name": first_nonblank(chunk, ["state_name"], default="not listed"),
        "geo_name": first_nonblank(chunk, ["geo_name"], default="not listed"),

        "industry_code": first_nonblank(chunk, ["industry_code", "naics_code"], default="not listed"),
        "industry_name": first_nonblank(chunk, ["industry_name"], default="not listed"),

        "metric_code": first_nonblank(chunk, ["metric_code"], default="not listed"),
        "metric_name": first_nonblank(chunk, ["metric_name"], default="not listed"),
        "metric_category": first_nonblank(chunk, ["metric_category"], default="not listed"),

        "value": value,
        "unit": first_nonblank(chunk, ["unit"], default="not listed")
    })

    compact["__sample_value__"] = compact["value"]

    return compact


# ------------------------------------------------------------
# Random rows by year, only rows with value > 0
# ------------------------------------------------------------
def random_rows_with_values_by_year(path, dataset_name, builder_func, rows_per_year=3, chunksize=50_000):
    print("\n" + "=" * 100)
    print(f"RANDOM ROWS WITH REAL VALUES BY YEAR: {dataset_name}")
    print("=" * 100)

    check_file(path)

    columns = get_columns(path)
    year_col = find_year_column(columns)

    print(f"Using year column: {year_col}")

    rng = np.random.default_rng(RANDOM_SEED)

    samples_by_year = {}
    row_counts = {}
    value_row_counts = {}

    total_rows_read = 0

    for chunk_number, chunk in enumerate(
        pd.read_csv(
            path,
            chunksize=chunksize,
            dtype=str
        ),
        start=1
    ):
        chunk_rows = len(chunk)

        chunk.insert(
            0,
            "__csv_row_number__",
            range(total_rows_read + 2, total_rows_read + 2 + chunk_rows)
        )

        chunk["__year_clean__"] = pd.to_numeric(chunk[year_col], errors="coerce")
        chunk = chunk.dropna(subset=["__year_clean__"])

        if chunk.empty:
            total_rows_read += chunk_rows
            continue

        chunk["__year_clean__"] = chunk["__year_clean__"].astype(int)

        counts = chunk["__year_clean__"].value_counts()
        for year, count in counts.items():
            row_counts[year] = row_counts.get(year, 0) + int(count)

        compact = builder_func(chunk)

        # Keep only rows with real value
        compact = compact[compact["__sample_value__"] > 0].copy()

        if compact.empty:
            total_rows_read += chunk_rows
            continue

        value_counts = compact["year"].value_counts()
        for year, count in value_counts.items():
            value_row_counts[year] = value_row_counts.get(year, 0) + int(count)

        compact["__random_key__"] = rng.random(len(compact))

        for year, group in compact.groupby("year"):
            group_sample = group.nsmallest(rows_per_year, "__random_key__")

            if year not in samples_by_year:
                samples_by_year[year] = group_sample.copy()
            else:
                combined = pd.concat([samples_by_year[year], group_sample], ignore_index=True)
                samples_by_year[year] = combined.nsmallest(rows_per_year, "__random_key__").copy()

        total_rows_read += chunk_rows

        if chunk_number % 10 == 0:
            print(f"Read {total_rows_read:,} rows so far...")

    print(f"\nDONE reading {total_rows_read:,} rows.")

    all_years = sorted(row_counts.keys())

    count_table = pd.DataFrame({
        "year": all_years,
        "total_rows_in_file": [row_counts.get(y, 0) for y in all_years],
        "rows_with_real_value": [value_row_counts.get(y, 0) for y in all_years]
    })

    count_table["rows_without_real_value"] = (
        count_table["total_rows_in_file"] - count_table["rows_with_real_value"]
    )

    print("\n" + "-" * 100)
    print("YEAR CHECK TABLE")
    print("-" * 100)
    display_clean(count_table)

    if not samples_by_year:
        print("\nNo rows with real value found.")
        return count_table, pd.DataFrame()

    sample_table = (
        pd.concat(samples_by_year.values(), ignore_index=True)
        .sort_values(["year", "__random_key__"])
        .reset_index(drop=True)
    )

    sample_table = sample_table.drop(columns=["__random_key__", "__sample_value__"], errors="ignore")

    print("\n" + "-" * 100)
    print(f"RANDOM ROWS WITH VALUES: {rows_per_year} ROWS PER YEAR")
    print("-" * 100)
    display_clean(sample_table)

    return count_table, sample_table


# ============================================================
# RUN DEGREE FILE
# ============================================================

degree_year_check, degree_random_value_rows = random_rows_with_values_by_year(
    DEGREE_FILE,
    "DEGREE FILE",
    build_degree_compact,
    rows_per_year=ROWS_PER_YEAR,
    chunksize=CHUNKSIZE
)


# ============================================================
# RUN COMPANY FILE
# ============================================================

company_year_check, company_random_value_rows = random_rows_with_values_by_year(
    COMPANY_FILE,
    "COMPANY FILE",
    build_company_compact,
    rows_per_year=ROWS_PER_YEAR,
    chunksize=CHUNKSIZE
)

In [ ]:
'''
---------------------------------------
YEAR CHECK TABLE
----------------------------------------------------------------------------------------------------
year	total_rows_in_file	rows_with_real_value	rows_without_real_value
0	1984	114727	114727	0
1	1985	116605	116605	0
2	1986	117479	117479	0
3	1987	126442	126442	0
4	1988	131154	131154	0
5	1989	132972	132972	0
6	1990	129518	129433	85
7	1991	129860	129860	0
8	1992	134877	133514	1363
9	1993	138445	136889	1556
10	1994	139740	139740	0
11	1995	163377	153145	10232
12	1996	158885	147805	11080
13	1997	159007	148976	10031
14	1998	201825	152896	48929
15	1999	153166	153166	0
16	2000	173137	155604	17533
17	2001	194569	171441	23128
18	2002	200336	175602	24734
19	2003	211859	181416	30443
20	2004	219809	186387	33422
21	2005	230217	191557	38660
22	2006	238450	194249	44201
23	2007	240422	197436	42986
24	2008	247127	201073	46054
25	2009	255934	206244	49690
26	2010	265728	212110	53618
27	2011	265825	219283	46542
28	2012	270217	223769	46448
29	2013	284937	227107	57830
30	2014	293274	229961	63313
31	2015	300263	231200	69063
32	2016	302043	230515	71528
33	2017	308882	231169	77713
34	2018	290149	232018	58131
35	2019	290972	229673	61299
36	2020	289833	231108	58725
37	2021	296365	232532	63833
38	2022	301055	235824	65231
39	2023	303460	236110	67350
40	2024	307707	237603	70104

----------------------------------------------------------------------------------------------------
RANDOM ROWS WITH VALUES: 3 ROWS PER YEAR
----------------------------------------------------------------------------------------------------
year	csv_row_number	unitid	cip_code	award_level_code	award_level_name	degree_group	field_group	field_subgroup	major_name	usable_degree_value	male_count	female_count	value_source
0	1984	10219	121363	181106	3	Associate	Associate	Unmapped / Nonstandard CIP Field (18)	Unmapped / Nonstandard CIP Subgroup (18.11)	Unmapped / Nonstandard CIP Major (181106)	1.0	0.0	1.0	crace15 + crace16
1	1984	62718	201645	180103	9	Doctor	Doctor	Unmapped / Nonstandard CIP Field (18)	Unmapped / Nonstandard CIP Subgroup (18.01)	Unmapped / Nonstandard CIP Major (180103)	2.0	0.0	2.0	crace15 + crace16
2	1984	18985	137078	170205	2	Certificate 1-2 years	Certificate/Other	Unmapped / Nonstandard CIP Field (17)	Unmapped / Nonstandard CIP Subgroup (17.02)	Unmapped / Nonstandard CIP Major (170205)	77.0	58.0	19.0	crace15 + crace16
3	1985	205060	151351	500704	7	Master	Master	VISUAL AND PERFORMING ARTS	Fine and Studio Arts	Fine and Studio Arts - unspecified major (5007040000)	6.0	0.0	6.0	crace15 + crace16
4	1985	176824	193900	179999	7	Master	Master	Unmapped / Nonstandard CIP Field (17)	Unmapped / Nonstandard CIP Subgroup (17.99)	Unmapped / Nonstandard CIP Major (179999)	6.0	5.0	1.0	crace15 + crace16
5	1985	139299	183743	181101	3	Associate	Associate	Unmapped / Nonstandard CIP Field (18)	Unmapped / Nonstandard CIP Subgroup (18.11)	Unmapped / Nonstandard CIP Major (181101)	139.0	4.0	135.0	crace15 + crace16
6	1986	283793	185129	131202	5	Bachelor	Bachelor	EDUCATION	Teacher Education and Professional Development, Specific Levels and Methods	Teacher Education and Professional Development, Specific Levels and Methods - unspecified major (1312020000)	27.0	1.0	26.0	crace15 + crace16
7	1986	235853	109907	430299	2	Certificate 1-2 years	Certificate/Other	HOMELAND SECURITY, LAW ENFORCEMENT, FIREFIGHTING AND RELATED PROTECTIVE SERVICES	Fire Protection	Fire Protection - unspecified major (4302990000)	1.0	1.0	0.0	crace15 + crace16
8	1986	317170	233921	260501.0	9	Doctor	Doctor	BIOLOGICAL AND BIOMEDICAL SCIENCES	Unmapped / Nonstandard CIP Subgroup (26050)	Unmapped / Nonstandard CIP Major (260501)	1.0	1.0	0.0	crace15 + crace16
9	1987	434247	201885	6.0601	7	Master	Master	Unmapped / Nonstandard CIP Field (06)	Unmapped / Nonstandard CIP Subgroup (06.06)	Unmapped / Nonstandard CIP Major (060601)	1.0	1.0	0.0	crace15 + crace16
10	1987	359459	114789	8.0799	3	Associate	Associate	Unmapped / Nonstandard CIP Field (08)	Unmapped / Nonstandard CIP Subgroup (08.07)	Unmapped / Nonstandard CIP Major (080799)	12.0	5.0	7.0	crace15 + crace16
11	1987	351526	102313	47.0105	3	Associate	Associate	MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS	Electrical/Electronics Maintenance and Repair Technologies/Technicians	Electrical/Electronics Maintenance and Repair Technologies/Technicians - unspecified major (470105)	7.0	5.0	2.0	crace15 + crace16
12	1988	480237	105525	40.0501	3	Associate	Associate	PHYSICAL SCIENCES	Chemistry	Chemistry - unspecified major (400501)	1.0	1.0	0.0	crace15 + crace16
13	1988	581289	218663	24.0101	3	Associate	Associate	LIBERAL ARTS AND SCIENCES, GENERAL STUDIES AND HUMANITIES	Liberal Arts and Sciences, General Studies and Humanities	Liberal Arts and Sciences, General Studies and Humanities - unspecified major (240101)	9.0	8.0	1.0	crace15 + crace16
14	1988	596094	234137	39.0601	7	Master	Master	THEOLOGY AND RELIGIOUS VOCATIONS	Theological and Ministerial Studies	Theological and Ministerial Studies - unspecified major (390601)	1.0	1.0	0.0	crace15 + crace16
15	1989	680131	172334	99.0	5	Bachelor	Bachelor	Other / Unknown CIP Field	Other / Unknown CIP Field	Unmapped / Nonstandard CIP Major (990000)	393.0	192.0	201.0	crace15 + crace16
16	1989	628040	226417	95.0	4	Certificate 2-4 years	Certificate/Other	Non-CIP / Unknown Field	Non-CIP / Unknown Field	Non-CIP / Unknown Field - unspecified major (950000)	3.0	1.0	2.0	crace15 + crace16
17	1989	606836	100724	31.0301	5	Bachelor	Bachelor	PARKS, RECREATION, LEISURE, FITNESS, AND KINESIOLOGY	Parks, Recreation, and Leisure Facilities Management	Parks, Recreation, and Leisure Facilities Management - unspecified major (310301)	4.0	2.0	2.0	crace15 + crace16
18	1990	795194	167534	15.0303	3	Associate	Associate	ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS	Electrical/Electronic Engineering Technologies/Technicians	Electrical/Electronic Engineering Technologies/Technicians - unspecified major (150303)	21.0	19.0	2.0	crace15 + crace16
19	1990	845186	218113	15.9999	3	Associate	Associate	ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS	Engineering/Engineering-Related Technologies/Technicians, Other	Engineering/Engineering-Related Technologies/Technicians, Other - unspecified major (159999)	18.0	16.0	2.0	crace15 + crace16
20	1990	838791	212832	42.0101	5	Bachelor	Bachelor	PSYCHOLOGY	Psychology, General	Psychology, General - unspecified major (420101)	8.0	2.0	6.0	crace15 + crace16
21	1991	897604	141574	14.0801	7	Master	Master	ENGINEERING	Civil Engineering	Civil Engineering - unspecified major (140801)	10.0	8.0	2.0	crace15 + crace16
22	1991	901041	145707	7.0602	3	Associate	Associate	Unmapped / Nonstandard CIP Field (07)	Unmapped / Nonstandard CIP Subgroup (07.06)	Unmapped / Nonstandard CIP Major (070602)	3.0	0.0	3.0	crace15 + crace16
23	1991	923308	168740	19.0503	5	Bachelor	Bachelor	FAMILY AND CONSUMER SCIENCES/HUMAN SCIENCES	Foods, Nutrition, and Related Services	Foods, Nutrition, and Related Services - unspecified major (190503)	3.0	1.0	2.0	crace15 + crace16
24	1992	1003551	243601	13.1316	5	Bachelor	Bachelor	EDUCATION	Teacher Education and Professional Development, Specific Subject Areas	Teacher Education and Professional Development, Specific Subject Areas - unspecified major (131316)	2.0	0.0	2.0	crace15 + crace16
25	1992	1021852	225511	50.0711	7	Master	Master	VISUAL AND PERFORMING ARTS	Fine and Studio Arts	Fine and Studio Arts - unspecified major (500711)	1.0	0.0	1.0	crace15 + crace16
26	1992	1129306	121619	52.0701	1	Certificate < 1 year	Certificate/Other	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Entrepreneurial and Small Business Operations	Entrepreneurial and Small Business Operations - unspecified major (520701)	3.0	2.0	1.0	crace15 + crace16
27	1993	1133893	100663	99.0	7	Master	Master	Other / Unknown CIP Field	Other / Unknown CIP Field	Unmapped / Nonstandard CIP Major (990000)	842.0	307.0	535.0	crace15 + crace16
28	1993	1170619	214111	52.0301	3	Associate	Associate	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Accounting and Related Services	Accounting and Related Services - unspecified major (520301)	11.0	6.0	5.0	crace15 + crace16
29	1993	1141188	240453	51.0706	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Health and Medical Administrative Services	Health and Medical Administrative Services - unspecified major (510706)	16.0	2.0	14.0	crace15 + crace16
30	1994	1325568	129020	38.0101	9	Doctor	Doctor	PHILOSOPHY AND RELIGIOUS STUDIES	Philosophy	Philosophy - unspecified major (380101)	1.0	1.0	0.0	crace15 + crace16
31	1994	1300087	158529	52.0401	2	Certificate 1-2 years	Certificate/Other	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Business Operations Support and Assistant Services	Business Operations Support and Assistant Services - unspecified major (520401)	2.0	0.0	2.0	crace15 + crace16
32	1994	1402437	238032	2.0201	7	Master	Master	Unmapped / Nonstandard CIP Field (02)	Unmapped / Nonstandard CIP Subgroup (02.02)	Unmapped / Nonstandard CIP Major (020201)	1.0	1.0	0.0	crace15 + crace16
33	1995	1565659	363633	11.0101	2	Certificate 1-2 years	Certificate/Other	COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES	Computer and Information Sciences, General	Computer and Information Sciences, General - unspecified major (110101)	2.0	0.0	2.0	crace15 + crace16
34	1995	1452319	148016	16.0905	5	Bachelor	Bachelor	FOREIGN LANGUAGES, LITERATURES, AND LINGUISTICS	Romance Languages, Literatures, and Linguistics	Romance Languages, Literatures, and Linguistics - unspecified major (160905)	1.0	1.0	0.0	crace15 + crace16
35	1995	1417998	109785	9.0101	5	Bachelor	Bachelor	COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS	Communication and Media Studies	Communication and Media Studies - unspecified major (090101)	24.0	14.0	10.0	crace15 + crace16
36	1996	1591788	130156	51.0801	2	Certificate 1-2 years	Certificate/Other	HEALTH PROFESSIONS AND RELATED PROGRAMS	Allied Health and Medical Assisting Services	Allied Health and Medical Assisting Services - unspecified major (510801)	51.0	1.0	50.0	crace15 + crace16
37	1996	1620963	157270	48.0105	3	Associate	Associate	PRECISION PRODUCTION	Unmapped / Nonstandard CIP Subgroup (48.01)	Unmapped / Nonstandard CIP Major (480105)	23.0	21.0	2.0	crace15 + crace16
38	1996	1711614	235097	40.0601	5	Bachelor	Bachelor	PHYSICAL SCIENCES	Geological and Earth Sciences/Geosciences	Geological and Earth Sciences/Geosciences - unspecified major (400601)	4.0	4.0	0.0	crace15 + crace16
39	1997	1801055	139755	51.0701	7	Master	Master	HEALTH PROFESSIONS AND RELATED PROGRAMS	Health and Medical Administrative Services	Health and Medical Administrative Services - unspecified major (510701)	9.0	3.0	6.0	crace15 + crace16
40	1997	1751383	182625	99.0	2	Certificate 1-2 years	Certificate/Other	Other / Unknown CIP Field	Other / Unknown CIP Field	Unmapped / Nonstandard CIP Major (990000)	4.0	0.0	4.0	crace15 + crace16
41	1997	1767571	169910	51.1601	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Nursing	Nursing - unspecified major (511601)	49.0	3.0	46.0	crace15 + crace16
42	1998	2094347	190150	45.0801	8	Post-master certificate	Certificate/Other	SOCIAL SCIENCES	Unmapped / Nonstandard CIP Subgroup (45.08)	Unmapped / Nonstandard CIP Major (450801)	51.0	38.0	13.0	crace15 + crace16
43	1998	2086905	182290	3.0101	5	Bachelor	Bachelor	NATURAL RESOURCES AND CONSERVATION	Natural Resources Conservation and Research	Natural Resources Conservation and Research - unspecified major (030101)	42.0	23.0	19.0	crace15 + crace16
44	1998	1987091	196015	47.0201	1	Certificate < 1 year	Certificate/Other	MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS	Heating, Air Conditioning, Ventilation and Refrigeration Maintenance Technology/Technician (HAC, HACR, HVAC, HVACR)	Heating, Air Conditioning, Ventilation and Refrigeration Maintenance Technology/Technician (HAC, HACR, HVAC, HVACR) ...	15.0	15.0	0.0	crace15 + crace16
45	1999	2204374	212832	27.0101	5	Bachelor	Bachelor	MATHEMATICS AND STATISTICS	Mathematics	Mathematics - unspecified major (270101)	3.0	1.0	2.0	crace15 + crace16
46	1999	2247100	429012	99.0	1	Certificate < 1 year	Certificate/Other	Other / Unknown CIP Field	Other / Unknown CIP Field	Unmapped / Nonstandard CIP Major (990000)	168.0	155.0	13.0	crace15 + crace16
47	1999	2228622	233639	51.1601	2	Certificate 1-2 years	Certificate/Other	HEALTH PROFESSIONS AND RELATED PROGRAMS	Nursing	Nursing - unspecified major (511601)	41.0	1.0	40.0	crace15 + crace16
48	2000	2288616	184782	40.0501	5	Bachelor	Bachelor	PHYSICAL SCIENCES	Chemistry	Chemistry - unspecified major (400501)	5.0	2.0	3.0	crace15 + crace16
49	2000	2348108	134097	13.1001	9	Doctor	Doctor	EDUCATION	Special Education and Teaching	Special Education and Teaching - unspecified major (131001)	1.0	0.0	1.0	crace15 + crace16
50	2000	2356266	125532	51.0601	3	Associate	Associate	HEALTH PROFESSIONS AND RELATED PROGRAMS	Dental Support Services and Allied Professions	Dental Support Services and Allied Professions - unspecified major (510601)	1.0	0.0	1.0	crace15 + crace16
51	2001	2611345	409315.0	51.1601	3	Associate	Associate	HEALTH PROFESSIONS AND RELATED PROGRAMS	Nursing	Nursing - unspecified major (511601)	40.0	12.0	28.0	crace15 + crace16
52	2001	2448283	185873	52.0301	2	Certificate 1-2 years	Certificate/Other	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Accounting and Related Services	Accounting and Related Services - unspecified major (520301)	1.0	1.0	0.0	crace15 + crace16
53	2001	2469895	170444	15.0303	3	Associate	Associate	ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS	Electrical/Electronic Engineering Technologies/Technicians	Electrical/Electronic Engineering Technologies/Technicians - unspecified major (150303)	2.0	2.0	0.0	crace15 + crace16
54	2002	2769441	218973.0	26.0101	5	Bachelor	Bachelor	Biological and Biomedical Sciences	Biology, General	Biology/Biological Sciences, General	40.0	18.0	22.0	CRACE15 + CRACE16
55	2002	2762375	215062.0	30.9999	7	Master	Master	Multi/Interdisciplinary Studies	Multi/Interdisciplinary Studies, Other	Multi-/Interdisciplinary Studies, Other	40.0	34.0	6.0	CRACE15 + CRACE16
56	2002	2721932	187985.0	50.0701	5	Bachelor	Bachelor	Visual and Performing Arts	Fine and Studio Art	Art/Art Studies, General	79.0	28.0	51.0	CRACE15 + CRACE16
57	2003	2816624	100751.0	13.1101	9	Doctor	Doctor	EDUCATION	Student Counseling and Personnel Services	Student Counseling and Personnel Services - unspecified major (131101)	6.0	1.0	5.0	crace15 + crace16
58	2003	2855646	137096.0	15.0401	3	Associate	Associate	ENGINEERING/ENGINEERING-RELATED TECHNOLOGIES/TECHNICIANS	Electromechanical Technologies/Technicians	Electromechanical Technologies/Technicians - unspecified major (150401)	7.0	4.0	3.0	crace15 + crace16
59	2003	2943542	198330.0	51.0909	2	Certificate 1-2 years	Certificate/Other	HEALTH PROFESSIONS AND RELATED PROGRAMS	Allied Health Diagnostic, Intervention, and Treatment Professions	Allied Health Diagnostic, Intervention, and Treatment Professions - unspecified major (510909)	14.0	1.0	13.0	crace15 + crace16
60	2004	3143299	186380.0	13.0408	9	Doctor	Doctor	Education	Educational Administration and Supervision	Elementary and Middle School Administration/Principalship	1.0	0.0	1.0	CRACE15 + CRACE16
61	2004	3070379	138956.0	11.0301	4	Certificate 2-4 years	Certificate/Other	Computer and Information Sciences and Support Services	Data Processing	Data Processing and Data Processing Technology/Technician	10.0	5.0	5.0	CRACE15 + CRACE16
62	2004	3183683	211981.0	3.0601	5	Bachelor	Bachelor	Natural Resources and Conservation	Wildlife and Wildlands Science and Management	Wildlife and Wildlands Science and Management	2.0	0.0	2.0	CRACE15 + CRACE16
63	2005	3453856	239248	52.0803	3	Associate	Associate	Business, Management, Marketing, and Related Support Services	Finance and Financial Management Services	Banking and Financial Support Services	1.0	1.0	0.0	CRACE15 + CRACE16
64	2005	3324359	157599	51.0899	3	Associate	Associate	Health Professions and Related Clinical Sciences	Allied Health and Medical Assisting Services	Allied Health and Medical Assisting Services, Other	5.0	0.0	5.0	CRACE15 + CRACE16
65	2005	3316028	153603	14.9999	5	Bachelor	Bachelor	Engineering	Engineering, Other	Engineering, Other	2.0	1.0	1.0	CRACE15 + CRACE16
66	2006	3633242	204936	13.0301	7	Master	Master	Education	Curriculum and Instruction	Curriculum and Instruction	12.0	2.0	10.0	CRACE15 + CRACE16
67	2006	3510599	127565	49.0104	5	Bachelor	Bachelor	Transportation and Materials Moving	Air Transportation	Aviation/Airway Management and Operations	18.0	14.0	4.0	CRACE15 + CRACE16
68	2006	3674371	228778	30.2201	5	Bachelor	Bachelor	Multi/Interdisciplinary Studies	Classical and Ancient Studies	Ancient Studies/Civilization	9.0	3.0	6.0	CRACE15 + CRACE16
69	2007	3842508	188182	99.0	7	Master	Master	Other / Unknown CIP Field	Unknown	Unknown	15.0	4.0	11.0	CRACE15 + CRACE16
70	2007	3884575	212425	51.0707	2	Certificate 1-2 years	Certificate/Other	Health Professions and Related Clinical Sciences	Health and Medical Administrative Services	Health Information/Medical Records Technology/Technician	14.0	1.0	13.0	CRACE15 + CRACE16
71	2007	3797531	159391	16.0102	9	Doctor	Doctor	Foreign languages, literatures, and Linguistics	Linguistic, Comparative, and Related Language Studies and Services	Linguistics	1.0	1.0	0.0	CRACE15 + CRACE16
72	2008	3960140	102553	19.0709	3	Associate	Associate	Family and Consumer Sciences/Human Sciences	Human Development, Family Studies, and Related Services	Child Care Provider/Assistant	12.0	0.0	12.0	degree_count
73	2008	3963906	105668	52.0101	2	Certificate 1-2 years	Certificate/Other	Business, Management, Marketing, and Related Support Services	Business/Commerce, General	Business/Commerce, General	7.0	5.0	2.0	degree_count
74	2008	4201936	449339	99.0	6	Post-baccalaureate certificate	Certificate/Other	Other / Unknown CIP Field	Unknown	Unknown	42.0	35.0	7.0	degree_count
75	2009	4243657	131496	16.0402	5	Bachelor	Bachelor	Foreign languages, literatures, and Linguistics	Slavic Languages, Literatures, and Linguistics, General	Russian Language and Literature	4.0	3.0	1.0	degree_count
76	2009	4224855	117247	10.0201	3	Associate	Associate	Communications Technologies/Technicians and Support Services	Audiovisual Communications Technologies/Technicians	Photographic and Film/Video Technology/Technician and Assistant	1.0	1.0	0.0	degree_count
77	2009	4424570	237011	16.0102	5	Bachelor	Bachelor	Foreign languages, literatures, and Linguistics	Linguistic, Comparative, and Related Language Studies and Services	Linguistics	16.0	6.0	10.0	degree_count
78	2010	4491180	123217	4.0901	2	Certificate 1-2 years	Certificate/Other	ARCHITECTURE AND RELATED SERVICES	Architectural Sciences and Technology	Architectural Technology/Technician	4.0	4.0	0.0	degree_count
79	2010	4703170	379463	9.0701	1	Certificate < 1 year	Certificate/Other	COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS	Radio, Television, and Digital Communication	Radio and Television	3.0	0.0	3.0	degree_count
80	2010	4564181	169248	52.0213	7	Master	Master	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Business Administration, Management and Operations	Organizational Leadership	139.0	69.0	70.0	degree_count
81	2011	4872099	197142	50.0101	5	Bachelor	Bachelor	VISUAL AND PERFORMING ARTS	Visual and Performing Arts, General	Visual and Performing Arts, General	6.0	4.0	2.0	degree_count
82	2011	4901860	211440	14.1101	7	Master	Master	ENGINEERING	Engineering Mechanics	Engineering Mechanics	1.0	0.0	1.0	degree_count
83	2011	4824977	168786	99.0	5	Bachelor	Bachelor	Other / Unknown CIP Field	Unknown	Unknown	43.0	15.0	28.0	degree_count
84	2012	5079635	162760	26.0101	5	Bachelor	Bachelor	BIOLOGICAL AND BIOMEDICAL SCIENCES	Biology, General	Biology/Biological Sciences, General	27.0	6.0	21.0	degree_count
85	2012	5078299	161484	51.0904	2	Certificate 1-2 years	Certificate/Other	HEALTH PROFESSIONS AND RELATED PROGRAMS	Allied Health Diagnostic, Intervention, and Treatment Professions	Emergency Medical Technology/Technician (EMT Paramedic)	8.0	7.0	1.0	degree_count
86	2012	5164593	207971	51.3801	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Registered Nursing, Nursing Administration, Nursing Research and Clinical Nursing	Registered Nursing/Registered Nurse	22.0	1.0	21.0	degree_count
87	2013	5435766	204024	13.1102	7	Master	Master	EDUCATION	Student Counseling and Personnel Services	College Student Counseling and Personnel Services	26.0	8.0	18.0	degree_count
88	2013	5457607	215099	42.281	5	Bachelor	Bachelor	PSYCHOLOGY	Clinical, Counseling and Applied Psychology	Health/Medical Psychology	14.0	5.0	9.0	degree_count
89	2013	5323784	146603	47.0604	3	Associate	Associate	MECHANIC AND REPAIR TECHNOLOGIES/TECHNICIANS	Vehicle Maintenance and Repair Technologies	Automobile/Automotive Mechanics Technology/Technician	13.0	11.0	2.0	degree_count
90	2014	5657921	170790	11.0201	3	Associate	Associate	COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES	Computer Programming	Computer Programming/Programmer, General	22.0	18.0	4.0	degree_count
91	2014	5778807	229355	50.0701	3	Associate	Associate	VISUAL AND PERFORMING ARTS	Fine and Studio Arts	Art/Art Studies, General	14.0	2.0	12.0	degree_count
92	2014	5769808	225788	11.1099	2	Certificate 1-2 years	Certificate/Other	COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES	Computer/Information Technology Administration and Management	Computer/Information Technology Services Administration and Management, Other	13.0	12.0	1.0	degree_count
93	2015	5974661	179566	51.3801	7	Master	Master	HEALTH PROFESSIONS AND RELATED PROGRAMS	Registered Nursing, Nursing Administration, Nursing Research and Clinical Nursing	Registered Nursing/Registered Nurse	2.0	0.0	2.0	degree_count
94	2015	5866361	121309	13.1001	7	Master	Master	EDUCATION	Special Education and Teaching	Special Education and Teaching, General	28.0	3.0	25.0	degree_count
95	2015	5882052	131496	45.0901	7	Master	Master	SOCIAL SCIENCES	International Relations and National Security Studies	International Relations and Affairs	247.0	140.0	107.0	degree_count
96	2016	6348434	213987	50.0409	5	Bachelor	Bachelor	VISUAL AND PERFORMING ARTS	Design and Applied Arts	Graphic Design	13.0	5.0	8.0	degree_count
97	2016	6356712	217475	52.0207	2	Certificate 1-2 years	Certificate/Other	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Business Administration, Management and Operations	Customer Service Management	3.0	3.0	0.0	degree_count
98	2016	6160623	114813	51.3801	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Registered Nursing, Nursing Administration, Nursing Research and Clinical Nursing	Registered Nursing/Registered Nurse	37.0	4.0	33.0	degree_count
99	2017	6673808	220862	14.0901	5	Bachelor	Bachelor	ENGINEERING	Computer Engineering	Computer Engineering, General	18.0	16.0	2.0	degree_count
100	2017	6509423	145770	99.0	17	Doctor - research/scholarship	Doctor	Other / Unknown CIP Field	Unknown	Unknown	28.0	9.0	19.0	degree_count
101	2017	6738669	454838	99.0	7	Master	Master	Other / Unknown CIP Field	Unknown	Unknown	12.0	2.0	10.0	degree_count
102	2018	6888977	183026	51.2201	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Public Health	Public Health, General	57.0	14.0	43.0	degree_count
103	2018	6959506	217518	13.1305	5	Bachelor	Bachelor	EDUCATION	Teacher Education and Professional Development, Specific Subject Areas	English/Language Arts Teacher Education	3.0	0.0	3.0	degree_count
104	2018	6903046	191719	51.1501	1	Certificate < 1 year	Certificate/Other	HEALTH PROFESSIONS AND RELATED PROGRAMS	Mental and Social Health Services and Allied Professions	Substance Abuse/Addiction Counseling	1.0	0.0	1.0	degree_count
105	2019	7043591	101693	50.0908	5	Bachelor	Bachelor	VISUAL AND PERFORMING ARTS	Music	Voice and Opera	2.0	0.0	2.0	degree_count
106	2019	7307007	367459	52.0101	3	Associate	Associate	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Business/Commerce, General	Business/Commerce, General	31.0	17.0	14.0	degree_count
107	2019	7329553	487472	52.0408	2	Certificate 1-2 years	Certificate/Other	BUSINESS, MANAGEMENT, MARKETING, AND RELATED SUPPORT SERVICES	Business Operations Support and Assistant Services	General Office Occupations and Clerical Services	28.0	7.0	21.0	degree_count
108	2020	7361017	121886	43.0206	21	Unknown	Unknown	HOMELAND SECURITY, LAW ENFORCEMENT, FIREFIGHTING AND RELATED PROTECTIVE SERVICES	Fire Protection	Wildland/Forest Firefighting and Investigation	7.0	6.0	1.0	degree_count
109	2020	7476368	186867	11.0102	6	Post-baccalaureate certificate	Certificate/Other	COMPUTER AND INFORMATION SCIENCES AND SUPPORT SERVICES	Computer and Information Sciences, General	Artificial Intelligence	1.0	1.0	0.0	degree_count
110	2020	7485539	191649	42.2804	17	Doctor - research/scholarship	Doctor	PSYCHOLOGY	Clinical, Counseling and Applied Psychology	Industrial and Organizational Psychology	5.0	2.0	3.0	degree_count
111	2021	7663627	129525	14.1401	7	Master	Master	ENGINEERING	Environmental/Environmental Health Engineering	Environmental/Environmental Health Engineering	4.0	1.0	3.0	degree_count
112	2021	7688607	144892	51.0701	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Health and Medical Administrative Services	Health/Health Care Administration/Management	11.0	2.0	9.0	degree_count
113	2021	7854452	227216	51.0204	5	Bachelor	Bachelor	HEALTH PROFESSIONS AND RELATED PROGRAMS	Communication Disorders Sciences and Services	Audiology/Audiologist and Speech-Language Pathology/Pathologist	1.0	1.0	0.0	degree_count
114	2022	8071901	188429	51.1508	7	Master	Master	HEALTH PROFESSIONS AND RELATED PROGRAMS	Mental and Social Health Services and Allied Professions	Mental Health Counseling/Counselor	26.0	6.0	20.0	degree_count
115	2022	8124764	212601	9.0903	5	Bachelor	Bachelor	COMMUNICATION, JOURNALISM, AND RELATED PROGRAMS	Public Relations, Advertising, and Applied Communication	Advertising	3.0	0.0	3.0	degree_count
116	2022	8025524	165662	99.0	5	Bachelor	Bachelor	Other / Unknown CIP Field	Unknown	Unknown	945.0	354.0	591.0	degree_count
117	2023	8351514	175786	1.0302	2	Certificate 1-2 years	Certificate/Other	AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED FIELDS	Agricultural Production Operations	Animal/Livestock Husbandry and Production	8.0	4.0	4.0	degree_count
118	2023	8508256	449463	12.0409	21	Unknown	Unknown	CULINARY, ENTERTAINMENT, AND PERSONAL SERVICES	Cosmetology and Related Personal Grooming Services	Aesthetician/Esthetician and Skin Care Specialist	65.0	0.0	65.0	degree_count
119	2023	8336637	169798	51.2305	6	Post-baccalaureate certificate	Certificate/Other	HEALTH PROFESSIONS AND RELATED PROGRAMS	Rehabilitation and Therapeutic Professions	Music Therapy/Therapist	1.0	0.0	1.0	degree_count
120	2024	8794723	240268	50.0913	21	Unknown	Unknown	VISUAL AND PERFORMING ARTS	Music	Music Technology	7.0	7.0	0.0	degree_count
121	2024	8576793	135267	46.0302	4	Certificate 2-4 years	Certificate/Other	CONSTRUCTION TRADES	Electrical and Power Transmission Installers	Electrician	33.0	33.0	0.0	degree_count
122	2024	8696302	196194	13.1319	2	Certificate 1-2 years	Certificate/Other	EDUCATION	Teacher Education and Professional Development, Specific Subject Areas	Technical Teacher Education	1.0	1.0	0.0	degree_count

====================================================================================================
RANDOM ROWS WITH REAL VALUES BY YEAR: COMPANY FILE
====================================================================================================
FOUND: /Users/YourUserName123/Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv
SIZE: 684.14 MB
Using year column: year
Read 500,000 rows so far...
Read 1,000,000 rows so far...
Read 1,500,000 rows so far...

DONE reading 1,911,932 rows.

----------------------------------------------------------------------------------------------------
YEAR CHECK TABLE
----------------------------------------------------------------------------------------------------
year	total_rows_in_file	rows_with_real_value	rows_without_real_value
0	1978	18246	17853	393
1	1979	20082	19460	622
2	1980	21948	20095	1853
3	1981	23788	21834	1954
4	1982	25656	23018	2638
5	1983	27485	24608	2877
6	1984	27475	26399	1076
7	1985	27481	25605	1876
8	1986	27487	25251	2236
9	1987	27513	25279	2234
10	1988	29333	26930	2403
11	1989	29350	27250	2100
12	1990	29366	27185	2181
13	1991	29372	26042	3330
14	1992	30000	27148	2852
15	1993	32656	30178	2478
16	1994	32741	30197	2544
17	1995	32692	30658	2034
18	1996	32728	29911	2817
19	1997	32726	30286	2440
20	1998	34553	32127	2426
21	1999	34522	31654	2868
22	2000	34554	32480	2074
23	2001	34573	31145	3428
24	2002	34595	30564	4031
25	2003	36424	32975	3449
26	2004	46260	43196	3064
27	2005	56211	52804	3407
28	2006	56195	53770	2425
29	2007	56153	52517	3636
30	2008	56074	52240	3834
31	2009	56068	51373	4695
32	2010	56092	51593	4499
33	2011	56124	52779	3345
34	2012	56120	53634	2486
35	2013	56123	53261	2862
36	2014	56246	53729	2517
37	2015	56198	53795	2403
38	2016	56156	53479	2677
39	2017	56176	53155	3021
40	2018	56258	53404	2854
41	2019	56354	53481	2873
42	2020	56328	52792	3536
43	2021	56530	52097	4433
44	2022	53825	52911	914
45	2023	51062	48861	2201
46	2024	16208	16199	9
47	2025	15686	15683	3
48	2026	6139	6139	0

----------------------------------------------------------------------------------------------------
RANDOM ROWS WITH VALUES: 3 ROWS PER YEAR
----------------------------------------------------------------------------------------------------
year	csv_row_number	source	dataset	geo_level	state_name	geo_name	industry_code	industry_name	metric_code	metric_name	metric_category	value	unit
0	1978	232685	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	31-33	not listed	job_creation_births	job_creation_births	startup_survival_failure_proxy	29335.000	count_or_rate_from_source
1	1978	232314	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	52	not listed	job_creation_births	job_creation_births	startup_survival_failure_proxy	3643.000	count_or_rate_from_source
2	1978	1414093	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	54	not listed	net_job_creation_rate	net_job_creation_rate	startup_survival_failure_proxy	200.000	count_or_rate_from_source
3	1979	455927	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	56	not listed	net_job_creation_rate	net_job_creation_rate	startup_survival_failure_proxy	11.191	count_or_rate_from_source
4	1979	770210	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	1.108	count_or_rate_from_source
5	1979	99731	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	61	not listed	emp	emp	startup_survival_failure_proxy	134519.000	count_or_rate_from_source
6	1980	629997	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firms	firms	startup_survival_failure_proxy	6401.000	count_or_rate_from_source
7	1980	56786	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	42	not listed	firms	firms	startup_survival_failure_proxy	3552.000	count_or_rate_from_source
8	1980	545203	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	22	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	4.000	count_or_rate_from_source
9	1981	102162	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	56	not listed	emp	emp	startup_survival_failure_proxy	104936.000	count_or_rate_from_source
10	1981	146226	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	21	not listed	denom	denom	startup_survival_failure_proxy	759.000	count_or_rate_from_source
11	1981	1269633	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	42	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	328903.000	count_or_rate_from_source
12	1982	147874	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	62	not listed	denom	denom	startup_survival_failure_proxy	25592.000	count_or_rate_from_source
13	1982	58134	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	31-33	not listed	firms	firms	startup_survival_failure_proxy	5728.000	count_or_rate_from_source
14	1982	954514	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	20722.000	count_or_rate_from_source
15	1983	1431333	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	23	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	9356.000	count_or_rate_from_source
16	1983	1002377	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation	job_creation	startup_survival_failure_proxy	1752.000	count_or_rate_from_source
17	1983	504264	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	21	not listed	reallocation_rate	reallocation_rate	startup_survival_failure_proxy	11.063	count_or_rate_from_source
18	1984	238885	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	42	not listed	job_creation_births	job_creation_births	startup_survival_failure_proxy	4004.000	count_or_rate_from_source
19	1984	149069	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	56	not listed	denom	denom	startup_survival_failure_proxy	39972.000	count_or_rate_from_source
20	1984	149765	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	48-49	not listed	denom	denom	startup_survival_failure_proxy	6834.000	count_or_rate_from_source
21	1985	106144	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	48-49	not listed	emp	emp	startup_survival_failure_proxy	30314.000	count_or_rate_from_source
22	1985	1225918	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	21	not listed	firms	firms	startup_survival_failure_proxy	2364.000	count_or_rate_from_source
23	1985	328525	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	71	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	7.637	count_or_rate_from_source
24	1986	62696	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	52	not listed	firms	firms	startup_survival_failure_proxy	3355.000	count_or_rate_from_source
25	1986	632357	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firms	firms	startup_survival_failure_proxy	2331.000	count_or_rate_from_source
26	1986	1278489	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	72	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	6.498	count_or_rate_from_source
27	1987	1184752	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	1228.000	count_or_rate_from_source
28	1987	1048882	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	2861.000	count_or_rate_from_source
29	1987	197278	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	51	not listed	job_creation	job_creation	startup_survival_failure_proxy	28810.000	count_or_rate_from_source
30	1988	286912	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	61	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	7139.000	count_or_rate_from_source
31	1988	633201	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firms	firms	startup_survival_failure_proxy	1825.000	count_or_rate_from_source
32	1988	108372	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	56	not listed	emp	emp	startup_survival_failure_proxy	38693.000	count_or_rate_from_source
33	1989	65711	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	23	not listed	firms	firms	startup_survival_failure_proxy	2701.000	count_or_rate_from_source
34	1989	1004838	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation	job_creation	startup_survival_failure_proxy	33326.000	count_or_rate_from_source
35	1989	199041	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	31-33	not listed	job_creation	job_creation	startup_survival_failure_proxy	1821.000	count_or_rate_from_source
36	1990	1432424	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	72	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	5692.000	count_or_rate_from_source
37	1990	17615	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	denom	denom	startup_survival_failure_proxy	3251373.000	count_or_rate_from_source
38	1990	1093707	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_rate	job_creation_rate	startup_survival_failure_proxy	16.155	count_or_rate_from_source
39	1991	1440447	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	22	not listed	firmdeath_emp	firmdeath_emp	startup_survival_failure_proxy	361.000	count_or_rate_from_source
40	1991	1186518	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	848.000	count_or_rate_from_source
41	1991	51766	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	21	not listed	reallocation_rate	reallocation_rate	startup_survival_failure_proxy	35.028	count_or_rate_from_source
42	1992	846617	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	net_job_creation_rate	net_job_creation_rate	startup_survival_failure_proxy	0.669	count_or_rate_from_source
43	1992	513076	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	11	not listed	reallocation_rate	reallocation_rate	startup_survival_failure_proxy	31.376	count_or_rate_from_source
44	1992	112285	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	55	not listed	emp	emp	startup_survival_failure_proxy	40016.000	count_or_rate_from_source
45	1993	730948	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_births	job_creation_births	startup_survival_failure_proxy	10253.000	count_or_rate_from_source
46	1993	557370	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	22	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	5.000	count_or_rate_from_source
47	1993	1165590	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	reallocation_rate	reallocation_rate	startup_survival_failure_proxy	34.380	count_or_rate_from_source
48	1994	959957	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	356074.000	count_or_rate_from_source
49	1994	1119402	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	net_job_creation	net_job_creation	startup_survival_failure_proxy	1129.000	count_or_rate_from_source
50	1994	683532	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	denom	denom	startup_survival_failure_proxy	143389.000	count_or_rate_from_source
51	1995	1895182	BLS BED/BDM	Business Employment Dynamics	state_or_us	U.S. totals	not listed	200000	Service-providing	08	Establishment Deaths	job_flows_openings_closings_gains_losses	993.000	level_from_source
52	1995	684185	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	denom	denom	startup_survival_failure_proxy	24638.000	count_or_rate_from_source
53	1995	560087	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	48-49	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	32.000	count_or_rate_from_source
54	1996	427570	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	31-33	not listed	net_job_creation	net_job_creation	startup_survival_failure_proxy	2485.000	count_or_rate_from_source
55	1996	661089	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	43712.000	count_or_rate_from_source
56	1996	24943	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	213439.000	count_or_rate_from_source
57	1997	1408181	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	61	not listed	net_job_creation	net_job_creation	startup_survival_failure_proxy	1275.000	count_or_rate_from_source
58	1997	1009108	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation	job_creation	startup_survival_failure_proxy	2528.000	count_or_rate_from_source
59	1997	893672	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	1117.000	count_or_rate_from_source
60	1998	661943	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	62065.000	count_or_rate_from_source
61	1998	208065	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	51	not listed	job_creation	job_creation	startup_survival_failure_proxy	765.000	count_or_rate_from_source
62	1998	20389	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation	job_creation	startup_survival_failure_proxy	29006.000	count_or_rate_from_source
63	1999	163851	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	31-33	not listed	denom	denom	startup_survival_failure_proxy	189278.000	count_or_rate_from_source
64	1999	1228523	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	56	not listed	firms	firms	startup_survival_failure_proxy	19076.000	count_or_rate_from_source
65	1999	208227	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	81	not listed	job_creation	job_creation	startup_survival_failure_proxy	2327.000	count_or_rate_from_source
66	2000	475479	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	44-45	not listed	net_job_creation_rate	net_job_creation_rate	startup_survival_failure_proxy	3.145	count_or_rate_from_source
67	2000	1340515	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	81	not listed	firms	firms	startup_survival_failure_proxy	30774.000	count_or_rate_from_source
68	2000	20460	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation	job_creation	startup_survival_failure_proxy	72448.000	count_or_rate_from_source
69	2001	387849	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	62	not listed	job_creation_rate	job_creation_rate	startup_survival_failure_proxy	9.453	count_or_rate_from_source
70	2001	963394	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	209945.000	count_or_rate_from_source
71	2001	210125	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	11	not listed	job_creation	job_creation	startup_survival_failure_proxy	90.000	count_or_rate_from_source
72	2002	11623	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	29607.000	count_or_rate_from_source
73	2002	166667	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	53	not listed	denom	denom	startup_survival_failure_proxy	14189.000	count_or_rate_from_source
74	2002	1170252	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	reallocation_rate	reallocation_rate	startup_survival_failure_proxy	17.588	count_or_rate_from_source
75	2003	640739	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firms	firms	startup_survival_failure_proxy	1140.000	count_or_rate_from_source
76	2003	688617	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	denom	denom	startup_survival_failure_proxy	62861.000	count_or_rate_from_source
77	2003	566921	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	48-49	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	38.000	count_or_rate_from_source
78	2004	1238397	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	44-45	not listed	emp	emp	startup_survival_failure_proxy	9472039.000	count_or_rate_from_source
79	2004	1148616	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	net_job_creation_rate	net_job_creation_rate	startup_survival_failure_proxy	1.114	count_or_rate_from_source
80	2004	1790207	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS53	NAICS53	BA_CBA	BA_CBA	business_applications_new_business_formation	4558.000	count_or_index_from_source
81	2005	214369	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	52	not listed	job_creation	job_creation	startup_survival_failure_proxy	15214.000	count_or_rate_from_source
82	2005	1514441	Census BFS	Business Formation Statistics	BFS geography	not listed	VT	TOTAL	TOTAL	BA_CBA	BA_CBA	business_applications_new_business_formation	106.000	count_or_index_from_source
83	2005	806267	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_rate	job_creation_rate	startup_survival_failure_proxy	14.011	count_or_rate_from_source
84	2006	714034	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation	job_creation	startup_survival_failure_proxy	5621.000	count_or_rate_from_source
85	2006	392673	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	62	not listed	job_creation_rate	job_creation_rate	startup_survival_failure_proxy	13.164	count_or_rate_from_source
86	2006	1384758	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	72	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	71213.000	count_or_rate_from_source
87	2007	1649051	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS42	NAICS42	BF_DUR8Q	BF_DUR8Q	business_applications_new_business_formation	1.980	count_or_index_from_source
88	2007	1786371	Census BFS	Business Formation Statistics	BFS geography	not listed	KS	TOTAL	TOTAL	BF_SBF4Q	BF_SBF4Q	business_applications_new_business_formation	226.000	count_or_index_from_source
89	2007	1614859	Census BFS	Business Formation Statistics	BFS geography	not listed	IL	TOTAL	TOTAL	BA_WBA	BA_WBA	business_applications_new_business_formation	3048.000	count_or_index_from_source
90	2008	614969	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	71	not listed	firmdeath_emp	firmdeath_emp	startup_survival_failure_proxy	589.000	count_or_rate_from_source
91	2008	1646278	Census BFS	Business Formation Statistics	BFS geography	not listed	OR	TOTAL	TOTAL	BA_HBA	BA_HBA	business_applications_new_business_formation	1234.000	count_or_index_from_source
92	2008	1282654	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	23	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	1.944	count_or_rate_from_source
93	2009	1851985	Census BFS	Business Formation Statistics	BFS geography	not listed	WE	TOTAL	TOTAL	BF_BF4Q	BF_BF4Q	business_applications_new_business_formation	5681.000	count_or_index_from_source
94	2009	1817534	Census BFS	Business Formation Statistics	BFS geography	not listed	DC	TOTAL	TOTAL	BF_PBF4Q	BF_PBF4Q	business_applications_new_business_formation	37.000	count_or_index_from_source
95	2009	1678246	Census BFS	Business Formation Statistics	BFS geography	not listed	IN	TOTAL	TOTAL	BF_BF8Q	BF_BF8Q	business_applications_new_business_formation	398.000	count_or_index_from_source
96	2010	1576188	Census BFS	Business Formation Statistics	BFS geography	not listed	MS	TOTAL	TOTAL	BF_SBF4Q	BF_SBF4Q	business_applications_new_business_formation	160.000	count_or_index_from_source
97	2010	1643469	Census BFS	Business Formation Statistics	BFS geography	not listed	UT	TOTAL	TOTAL	BF_BF8Q	BF_BF8Q	business_applications_new_business_formation	294.000	count_or_index_from_source
98	2010	1815998	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS81	NAICS81	BF_PBF8Q	BF_PBF8Q	business_applications_new_business_formation	1695.000	count_or_index_from_source
99	2011	1674999	Census BFS	Business Formation Statistics	BFS geography	not listed	UT	TOTAL	TOTAL	BF_BF8Q	BF_BF8Q	business_applications_new_business_formation	408.000	count_or_index_from_source
100	2011	1848342	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS55	NAICS55	BA_HBA	BA_HBA	business_applications_new_business_formation	518.000	count_or_index_from_source
101	2011	1231213	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	61	not listed	firms	firms	startup_survival_failure_proxy	5741.000	count_or_rate_from_source
102	2012	1444526	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	52	not listed	firmdeath_emp	firmdeath_emp	startup_survival_failure_proxy	3932.000	count_or_rate_from_source
103	2012	1571850	Census BFS	Business Formation Statistics	BFS geography	not listed	SC	TOTAL	TOTAL	BA_CBA	BA_CBA	business_applications_new_business_formation	243.000	count_or_index_from_source
104	2012	670243	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	6882.000	count_or_rate_from_source
105	2013	354741	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	22	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	0.484	count_or_rate_from_source
106	2013	1536330	Census BFS	Business Formation Statistics	BFS geography	not listed	NM	TOTAL	TOTAL	BF_BF8Q	BF_BF8Q	business_applications_new_business_formation	138.000	count_or_index_from_source
107	2013	1671721	Census BFS	Business Formation Statistics	BFS geography	not listed	WA	TOTAL	TOTAL	BF_BF8Q	BF_BF8Q	business_applications_new_business_formation	783.000	count_or_index_from_source
108	2014	1569200	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS56	NAICS56	BF_PBF4Q	BF_PBF4Q	business_applications_new_business_formation	1115.000	count_or_index_from_source
109	2014	1083965	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	0.727	count_or_rate_from_source
110	2014	1636205	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS55	NAICS55	BA_CBA	BA_CBA	business_applications_new_business_formation	361.000	count_or_index_from_source
111	2015	179822	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	62	not listed	denom	denom	startup_survival_failure_proxy	65328.000	count_or_rate_from_source
112	2015	1267601	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	72	not listed	job_creation_births	job_creation_births	startup_survival_failure_proxy	20688.000	count_or_rate_from_source
113	2015	1807247	Census BFS	Business Formation Statistics	BFS geography	not listed	AR	TOTAL	TOTAL	BA_WBA	BA_WBA	business_applications_new_business_formation	249.000	count_or_index_from_source
114	2016	1108689	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_rate	job_creation_rate	startup_survival_failure_proxy	200.000	count_or_rate_from_source
115	2016	860752	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	net_job_creation_rate	net_job_creation_rate	startup_survival_failure_proxy	0.768	count_or_rate_from_source
116	2016	1232382	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	72	not listed	firms	firms	startup_survival_failure_proxy	39025.000	count_or_rate_from_source
117	2017	1700237	Census BFS	Business Formation Statistics	BFS geography	not listed	MN	TOTAL	TOTAL	BF_PBF8Q	BF_PBF8Q	business_applications_new_business_formation	422.000	count_or_index_from_source
118	2017	623265	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	42	not listed	firmdeath_emp	firmdeath_emp	startup_survival_failure_proxy	3736.000	count_or_rate_from_source
119	2017	696978	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	denom	denom	startup_survival_failure_proxy	37469.000	count_or_rate_from_source
120	2018	767035	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	4087.000	count_or_rate_from_source
121	2018	1768533	Census BFS	Business Formation Statistics	BFS geography	not listed	NM	TOTAL	TOTAL	BF_DUR4Q	BF_DUR4Q	business_applications_new_business_formation	0.830	count_or_index_from_source
122	2018	1803082	Census BFS	Business Formation Statistics	BFS geography	not listed	NH	TOTAL	TOTAL	BF_SBF4Q	BF_SBF4Q	business_applications_new_business_formation	103.000	count_or_index_from_source
123	2019	674313	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	emp	emp	startup_survival_failure_proxy	8489.000	count_or_rate_from_source
124	2019	1525712	Census BFS	Business Formation Statistics	BFS geography	not listed	IA	TOTAL	TOTAL	BA_CBA	BA_CBA	business_applications_new_business_formation	149.000	count_or_index_from_source
125	2019	1661426	Census BFS	Business Formation Statistics	BFS geography	not listed	GA	TOTAL	TOTAL	BA_WBA	BA_WBA	business_applications_new_business_formation	1578.000	count_or_index_from_source
126	2020	1065810	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	24560.000	count_or_rate_from_source
127	2020	1884592	BLS BED/BDM	Business Employment Dynamics	state_or_us	U.S. totals	not listed	200090	Leisure and hospitality	04	Gross Job Losses	job_flows_openings_closings_gains_losses	234.000	level_from_source
128	2020	1065874	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	job_creation_continuers	job_creation_continuers	startup_survival_failure_proxy	1813.000	count_or_rate_from_source
129	2021	451320	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	11	not listed	net_job_creation	net_job_creation	startup_survival_failure_proxy	11.000	count_or_rate_from_source
130	2021	362693	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	81	not listed	job_creation_rate_births	job_creation_rate_births	startup_survival_failure_proxy	3.338	count_or_rate_from_source
131	2021	1863078	BLS BED/BDM	Business Employment Dynamics	state_or_us	U.S. totals	not listed	100000	Goods-producing	03	Openings	job_flows_openings_closings_gains_losses	195.000	level_from_source
132	2022	1521399	Census BFS	Business Formation Statistics	BFS geography	not listed	MO	TOTAL	TOTAL	BF_BF4Q	BF_BF4Q	business_applications_new_business_formation	453.000	count_or_index_from_source
133	2022	907543	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	firmdeath_firms	firmdeath_firms	startup_survival_failure_proxy	491.000	count_or_rate_from_source
134	2022	142113	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	52	not listed	emp	emp	startup_survival_failure_proxy	82237.000	count_or_rate_from_source
135	2023	142296	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	22	not listed	emp	emp	startup_survival_failure_proxy	7485.000	count_or_rate_from_source
136	2023	1519902	Census BFS	Business Formation Statistics	BFS geography	not listed	TX	TOTAL	TOTAL	BA_HBA	BA_HBA	business_applications_new_business_formation	18978.000	count_or_index_from_source
137	2023	700998	Census BDS	Business Dynamics Statistics	varies_by_bds_file	not listed	not listed	not listed	not listed	denom	denom	startup_survival_failure_proxy	262132.000	count_or_rate_from_source
138	2024	1758844	Census BFS	Business Formation Statistics	BFS geography	not listed	PA	TOTAL	TOTAL	BF_PBF4Q	BF_PBF4Q	business_applications_new_business_formation	659.000	count_or_index_from_source
139	2024	1793363	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS54	NAICS54	BA_WBA	BA_WBA	business_applications_new_business_formation	4157.000	count_or_index_from_source
140	2024	1518329	Census BFS	Business Formation Statistics	BFS geography	not listed	LA	TOTAL	TOTAL	BA_BA	BA_BA	business_applications_new_business_formation	5800.000	count_or_index_from_source
141	2025	1518033	Census BFS	Business Formation Statistics	BFS geography	not listed	MI	TOTAL	TOTAL	BF_SBF4Q	BF_SBF4Q	business_applications_new_business_formation	538.000	count_or_index_from_source
142	2025	1653366	Census BFS	Business Formation Statistics	BFS geography	not listed	US	TOTAL	TOTAL	BF_SBF4Q	BF_SBF4Q	business_applications_new_business_formation	25391.000	count_or_index_from_source
143	2025	1827458	Census BFS	Business Formation Statistics	BFS geography	not listed	NM	TOTAL	TOTAL	BF_SBF4Q	BF_SBF4Q	business_applications_new_business_formation	187.000	count_or_index_from_source
144	2026	1861205	U.S. Courts F-2	Bankruptcy filings	US total	not listed	United States	not listed	not listed	f2_value_03	F-2 Total row numeric value 03	bankruptcy_filings	312.000	filings
145	2026	1516964	Census BFS	Business Formation Statistics	BFS geography	not listed	MS	TOTAL	TOTAL	BF_SBF8Q	BF_SBF8Q	business_applications_new_business_formation	216.000	count_or_index_from_source
146	2026	1585426	Census BFS	Business Formation Statistics	BFS geography	not listed	US	NAICS71	NAICS71	BF_PBF4Q	BF_PBF4Q	business_applications_new_business_formation	628.000	count_or_index_from_source
'''

1984–2002:
Use IPEDS degree values from old columns
+
Use Census BDS company proxy

2003–2023:
Use IPEDS degree values
+
Use BLS actual job-flow if available
+
Use BDS when BLS is missing

Do not use 2024–2026 for final clean chart
because company data becomes partial/mixed.


# =========================
# FILE PATHS
# =========================
degree_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

company_file = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/business_startup_ALL_IN_ONE.csv"



Prompt text:


In [ ]:
'''

'''

# Success 4 - degree
| Code purpose        | Column name        | Meaning                                                  |
| ------------------- | ------------------ | -------------------------------------------------------- |
| Main year column    | `year`             | Groups data by year                                      |
| Number/value column | `degree_count`     | The degree amount/count being added                      |
| Category 1          | `major_name`       | Major name                                               |
| Category 2          | `field_group`      | Big field group                                          |
| Category 3          | `field_subgroup`   | Smaller field group                                      |
| Category 4          | `degree_group`     | Degree type/group                                        |
| Category 5          | `award_level_name` | Award level, like certificate, associate, bachelor, etc. |
| Category 6          | `cip_code`         | CIP code for the major/field                             |


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ============================================================
# DEGREE COUNT BY YEAR + ALL CATEGORIES
# - NO TOP N
# - NO OTHER BUCKET
# - REMOVE UNKNOWN / BLANK / NAN
# - PRINT TABLE
# - SAVE FULL CSV
# - SAVE IMAGE
# ============================================================

CFG = {
    "DEGREE_FILE": Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv",
    "OUTPUT_DIR": Path.home() / "Downloads" / "degree_count_all_table_and_image",

    "START_YEAR": 2020,
    "END_YEAR": 2025,   # 2025 will be 0 if file stops at 2024

    "MAIN_COLUMN": "year",
    "VALUE_COLUMN": "degree_count",

    "OTHER_COLUMNS": [
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "NEEDED_COLUMNS": [
        "year",
        "major_name",
        "field_group",
        "field_subgroup",
        "degree_count",
        "degree_group",
        "award_level_name",
        "cip_code",
    ],

    "CHUNKSIZE": 100_000,

    # print
    "PRINT_ALL_ROWS": True,   # True = print full table
    "PRINT_MAX_ROWS": None,   # None = unlimited

    # chart
    "FIGSIZE": (20, 10),
    "DPI": 200,
    "SHOW_LEGEND": True,      # can be too large for big columns
    "LEGEND_FONT_SIZE": 8,
}

CFG["OUTPUT_DIR"].mkdir(parents=True, exist_ok=True)

try:
    from IPython.display import display
except ImportError:
    display = print

# ============================================================
# PANDAS DISPLAY
